# Session 1 — NumPy core: arrays, indexing, shapes

Companion to the plan in [../numpy_pytorch_schedule.md](../numpy_pytorch_schedule.md). Goal: stop thinking in loops, start thinking in **shapes**. Run each cell, read its output, and change the numbers to see what moves.

An `ndarray` is three things: a flat buffer of numbers, a `dtype` (how to read each number), and a `shape` (how to index the buffer as an N-D grid). Almost every op either *reads the buffer differently* (reshape/transpose — cheap, no copy) or *computes a new one* (arithmetic, reductions).

In [1]:
import numpy as np

x = np.arange(12).reshape(3, 4)     # shape (3,4): 3 rows, 4 cols
print(x)
print("shape:", x.shape, "| dtype:", x.dtype, "| ndim:", x.ndim)

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
shape: (3, 4) | dtype: int64 | ndim: 2


## Creation and inspection

Always know the `shape` and `dtype` of what you hold. In DL, `float32` is the working default; index arrays are `int64`.

In [2]:
print(np.zeros((2, 3)))
print(np.ones((2, 3)))
print(np.arange(0, 10, 2))          # [0 2 4 6 8]
print(np.linspace(0, 1, 5))         # 5 evenly spaced points in [0,1]
print(np.random.randn(2, 3))        # standard normal, shape (2,3)

[[0. 0. 0.]
 [0. 0. 0.]]
[[1. 1. 1.]
 [1. 1. 1.]]
[0 2 4 6 8]
[0.   0.25 0.5  0.75 1.  ]
[[-0.19832073  0.88175523  0.94645087]
 [-0.74227397 -0.61886369  0.10968823]]


## Indexing and slicing

Three flavors, all worth fluency: basic slices, **boolean masks**, and **fancy indexing** (pick rows/elements by an index array). Masks and fancy indexing are how you gather without loops — e.g. selecting the logits of the correct tokens.

In [3]:
x = np.arange(12).reshape(3, 4)
print("row 0     :", x[0])          # -> shape (4,)
print("col 1     :", x[:, 1])       # -> shape (3,)
print("block     :\n", x[0:2, 1:3])# -> shape (2,2)
print("mask x>5  :", x[x > 5])      # 1-D of matching elements
print("rows 0,2  :\n", x[[0, 2]])  # fancy indexing -> shape (2,4)

row 0     : [0 1 2 3]
col 1     : [1 5 9]
block     :
 [[1 2]
 [5 6]]
mask x>5  : [ 6  7  8  9 10 11]
rows 0,2  :
 [[ 0  1  2  3]
 [ 8  9 10 11]]


## Reshaping — reorder vs. reinterpret

This distinction trips people up:
- **`reshape`** reinterprets the *same data* under a new shape (row-major order preserved).
- **`transpose` / `.T`** *reorders* axes — it changes which element is "next" in memory.

In [4]:
x = np.arange(6)                    # [0 1 2 3 4 5]
print("reshape(2,3):\n", x.reshape(2, 3))     # same order, regrouped
print("reshape then .T:\n", x.reshape(2, 3).T) # axes swapped, order changed
print("add axis:", x[np.newaxis, :].shape)      # (1,6)  (same as x[None,:])
print("squeeze :", x.reshape(2, 3, 1).squeeze().shape)  # (2,3)

reshape(2,3):
 [[0 1 2]
 [3 4 5]]
reshape then .T:
 [[0 3]
 [1 4]
 [2 5]]
add axis: (1, 6)
squeeze : (2, 3)


### How row-major layout works (why `reshape` is free)

Under the hood an array is a **flat 1-D buffer** plus a `shape` and **strides**. NumPy/PyTorch store data **row-major**: the *last axis varies fastest* — you walk one full row before moving to the next. Picture the multi-index as a **mixed-radix number read right-to-left** (an odometer: the rightmost index spins fastest, and when it rolls over the next-left index ticks).

Each axis has a "place value" = its **stride** (how many buffer slots to jump to advance that index by 1), and the rule is:

> **stride of an axis = the product of the sizes of all axes to its right.**

Equivalently, build strides **right-to-left as a running product**: the last axis has stride `1`, and each axis to the left is `(size to its right) × (stride to its right)` — i.e. each stride is a *multiple of the value to its right*. The flat offset of an element is then just `Σ index[i] × stride[i]`, exactly like `hundreds·100 + tens·10 + units·1`.

This is *why* **`reshape` is free** (no copy): the numbers are already in row-major order in the buffer, so regrouping the shape only recomputes strides — nothing moves. It's also why **`transpose` reorders**: it swaps strides without moving data, so the last axis is no longer stride-1 (the array becomes *non-contiguous*, which is why a later `view` can error — Session 3).

In [ ]:
import numpy as np
x = np.arange(24).reshape(2, 3, 4)                     # shape (2,3,4)
print("flat buffer:", x.ravel().tolist())              # 0..23, row-major (last axis fastest)
print("element strides:", tuple(s // x.itemsize for s in x.strides))   # (12, 4, 1) = (3*4, 4, 1)
#   right-to-left running product: last=1, then 1*4=4, then 4*3=12  -> (12, 4, 1)
i, j, k = 1, 2, 3
print(f"offset(1,2,3) = 1*12 + 2*4 + 3*1 = {i*12 + j*4 + k} -> value {x[i, j, k]}")   # 23

## Reductions along axes — the #1 bug source

`sum/mean/max/argmax/std` collapse axes. **The axis you name is the one that disappears.** `keepdims=True` keeps it as size-1 so the result **broadcasts back** — essential for normalization.

In [5]:
x = np.arange(12).reshape(3, 4)
print("sum()            :", x.sum())              # scalar
print("sum(axis=0)      :", x.sum(axis=0), "shape", x.sum(axis=0).shape)   # (4,) column sums
print("sum(axis=1)      :", x.sum(axis=1), "shape", x.sum(axis=1).shape)   # (3,) row sums
print("sum(axis=1,keep) shape:", x.sum(axis=1, keepdims=True).shape)       # (3,1)

row_mean = x.mean(axis=1, keepdims=True)   # (3,1)
print("centered (broadcasts):\n", x - row_mean)  # (3,4) - (3,1) works

sum()            : 66
sum(axis=0)      : [12 15 18 21] shape (4,)
sum(axis=1)      : [ 6 22 38] shape (3,)
sum(axis=1,keep) shape: (3, 1)
centered (broadcasts):
 [[-1.5 -0.5  0.5  1.5]
 [-1.5 -0.5  0.5  1.5]
 [-1.5 -0.5  0.5  1.5]]


> **Habit:** use `keepdims=True` whenever you'll subtract/divide the reduction back — otherwise `x - x.mean(axis=1)` misbroadcasts.

## Self-check

1. `x` is `(3,4)`. Shapes of `x.sum(axis=0)`, `x.sum(axis=1)`, `x.sum(axis=1, keepdims=True)`?
2. Do `x.reshape(4,3)` and `x.T` hold the same numbers in the same order?
3. Why does `x - x.mean(axis=1)` often fail while `x - x.mean(axis=1, keepdims=True)` works?

**Answers.** (1) `(4,)`, `(3,)`, `(3,1)` — the named axis vanishes; keepdims keeps it size-1. (2) No — `reshape` regroups in row-major order; `.T` swaps axes so element order changes (`x.T[i,j]=x[j,i]`). (3) `mean(axis=1)` is `(3,)`; broadcasting aligns trailing dims so `(3,4)-(3,)` mismatches 4 vs 3. `keepdims` gives `(3,1)`, which broadcasts down the columns.

## Exercise — standardize each row (LayerNorm's core op)

Make `x = np.random.randn(4, 5)`. Without any loop, compute each **row's** mean and std, standardize `x_norm = (x - mean)/std`, and confirm per-row mean ~0 and std ~1.

In [ ]:
x = np.random.randn(4, 5)
mean = x.mean(axis=1, keepdims=True)          # (4,1)
std  = x.std(axis=1, keepdims=True)           # (4,1)
x_norm = (x - mean) / (std + 1e-5)            # broadcast over columns
print("row means:", np.round(x_norm.mean(axis=1), 6))   # ~[0 0 0 0]
print("row stds :", np.round(x_norm.std(axis=1), 4))    # ~[1 1 1 1]

The `keepdims=True` on both reductions makes the per-row broadcast work; the `1e-5` is the same numerical-stability `eps` LayerNorm uses. You'll rebuild this exact op inside a `nn.LayerNorm` in Session 5.